In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%run ./_local_config

### Read bronze

In [0]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(container="bronze", blob="erp/battery/battery.json")

stream = blob_client.download_blob().readall()
text = stream.decode("utf-8-sig")
pages = [json.loads(line) for line in text.splitlines() if line.strip()]

all_records = []
for page in pages:
    all_records.extend(page["value"])

bronze = pd.DataFrame(all_records)
print(bronze.shape)
bronze.head()

### Clean to silver

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver

silver = clean_to_silver(bronze)
print(silver.shape)
silver.head()

### Sanity checks

In [0]:
print(silver["documentType"].value_counts())
print(silver.groupby("documentType")["net_units"].describe())
print(f"Total profit: {silver['profit'].sum():,.2f}")

### Vehicle type

In [0]:
unique_vehicle_types = sorted(silver["vehicle_type"].dropna().unique())
vehicle_type_lookup = pd.DataFrame({
    "vehicle_type_id": [f"V{i}" for i in range(1, len(unique_vehicle_types) + 1)],
    "vehicle_type": unique_vehicle_types
})
print(vehicle_type_lookup)

### Brand counts

In [0]:
brand_counts = silver.groupby(["brand_code", "brand_description"]).size().reset_index(name="count")
print(brand_counts.sort_values("count", ascending=False).to_string())

### Save silver

In [0]:
silver_json = silver.to_json(orient="records", date_format="iso", lines=False)
blob_client = blob_service.get_blob_client(container="silver", blob="erp/battery/battery_clean.json")
blob_client.upload_blob(silver_json, overwrite=True)
print(f"Saved {len(silver)} rows to silver/erp/battery/battery_clean.json")

### Check data reliability

In [0]:
recent_90 = silver[silver["posting_date"] >= silver["posting_date"].max() - pd.Timedelta(days=90)]
daily_counts_90 = recent_90.groupby(recent_90["posting_date"].dt.date).size()
print(daily_counts_90.tail(30))

### Trim to reliable data

In [0]:
cutoff_date = pd.Timestamp("2026-06-12")
silver_trimmed = silver[silver["posting_date"] <= cutoff_date].copy()
print(f"Original: {len(silver)}, Trimmed: {len(silver_trimmed)}")

### Build weekly gold

In [0]:
from src.transform.build_gold_features import build_gold_overall_weekly, build_gold_overall_monthly

gold_weekly = build_gold_overall_weekly(silver_trimmed)
print(gold_weekly.shape)
gold_weekly.tail(10)

###  Build monthly gold

In [0]:
monthly_cutoff = pd.Timestamp("2026-05-31")
silver_trimmed_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()

gold_monthly = build_gold_overall_monthly(silver_trimmed_monthly)
print(gold_monthly.tail(5)[["month_start", "total_units_sold"]])

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(gold_weekly["week_start"], gold_weekly["total_units_sold"])
axes[0].set_title("Weekly Net Battery Sales (3 Brands, Trimmed)")
axes[0].set_ylabel("Net Units")

axes[1].plot(gold_monthly["month_start"], gold_monthly["total_units_sold"], marker="o")
axes[1].set_title("Monthly Net Battery Sales (3 Brands, Trimmed)")
axes[1].set_ylabel("Net Units")

plt.tight_layout()
plt.show()

### Save both gold datasets

In [0]:
import io

for name, df in [("weekly", gold_weekly), ("monthly", gold_monthly)]:
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)
    blob_client = blob_service.get_blob_client(container="gold", blob=f"erp/battery/phase1_overall_{name}.parquet")
    blob_client.upload_blob(buffer, overwrite=True)
    print(f"Saved {len(df)} rows to gold/erp/battery/phase1_overall_{name}.parquet")

### Weekly model: train/test split

In [0]:
feature_cols_weekly = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w"]
target_col = "total_units_sold"

model_data_weekly = gold_weekly.dropna(subset=feature_cols_weekly + [target_col]).copy()
split_idx = int(len(model_data_weekly) * 0.8)
train_weekly = model_data_weekly.iloc[:split_idx]
test_weekly = model_data_weekly.iloc[split_idx:]

print(f"Train: {len(train_weekly)} weeks, Test: {len(test_weekly)} weeks")

X_train_w, y_train_w = train_weekly[feature_cols_weekly], train_weekly[target_col]
X_test_w, y_test_w = test_weekly[feature_cols_weekly], test_weekly[target_col]

In [0]:
#%pip install lightgbm prophet

In [0]:
#%restart_python

### Weekly model: train + evaluate

In [0]:
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

with mlflow.start_run(run_name="phase1_overall_weekly_netunits"):
    params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6}
    mlflow.log_params(params)

    model_w = lgb.LGBMRegressor(**params)
    model_w.fit(X_train_w, y_train_w)

    preds_w = model_w.predict(X_test_w)

    mae_w = mean_absolute_error(y_test_w, preds_w)
    rmse_w = mean_squared_error(y_test_w, preds_w) ** 0.5
    wape_w = wape(y_test_w, preds_w)

    mlflow.log_metric("mae", mae_w)
    mlflow.log_metric("rmse", rmse_w)
    mlflow.log_metric("wape", wape_w)
    mlflow.lightgbm.log_model(model_w, name="model")

    print(f"Weekly Model — MAE: {mae_w:.2f}   RMSE: {rmse_w:.2f}   WAPE: {wape_w:.3%}")

    baseline_preds_w = X_test_w["rolling_avg_4w"]
    print(f"Weekly Baseline — MAE: {mean_absolute_error(y_test_w, baseline_preds_w):.2f}   WAPE: {wape(y_test_w, baseline_preds_w):.3%}")

In [0]:
plt.figure(figsize=(16, 5))
plt.plot(test_weekly["week_start"], y_test_w.values, label="Actual", marker="o")
plt.plot(test_weekly["week_start"], preds_w, label="Predicted", marker="x")
plt.legend()
plt.title("Weekly Model — Actual vs Predicted (Test Set)")
plt.ylabel("Net Units")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Weekly: predict next week

In [0]:
X_all_w = model_data_weekly[feature_cols_weekly]
y_all_w = model_data_weekly[target_col]

final_weekly_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
final_weekly_model.fit(X_all_w, y_all_w)

last_week_start = gold_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

lag_4w_value = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
lag_4w_value = lag_4w_value[0] if len(lag_4w_value) > 0 else None

next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "lag_4w": lag_4w_value,
    "rolling_avg_4w": gold_weekly["total_units_sold"].tail(4).mean(),
}])

next_week_prediction = final_weekly_model.predict(next_week_features[feature_cols_weekly])
print(f"Predicted net units for week starting {next_week_start.date()}: {next_week_prediction[0]:.0f}")

### Monthly model

In [0]:
from prophet import Prophet

monthly_prophet_df = gold_monthly[["month_start", "total_units_sold"]].rename(
    columns={"month_start": "ds", "total_units_sold": "y"}
)

train_monthly = monthly_prophet_df.iloc[:-3]
test_monthly = monthly_prophet_df.iloc[-3:]

m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m_test.fit(train_monthly)

future_test = m_test.make_future_dataframe(periods=3, freq="MS")
forecast_test = m_test.predict(future_test)

test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values
print(f"Monthly Prophet WAPE: {wape(test_monthly['y'].values, test_preds):.3%}")

In [0]:
plt.figure(figsize=(10, 5))
plt.plot(test_monthly["ds"], test_monthly["y"], label="Actual", marker="o")
plt.plot(test_monthly["ds"], test_preds, label="Predicted", marker="x")
plt.legend()
plt.title("Monthly Prophet — Actual vs Predicted (Last 3 Months, Backtest)")
plt.ylabel("Net Units")
plt.tight_layout()
plt.show()

### Monthly: 3-month future forecast

In [0]:
m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m_final.fit(monthly_prophet_df)

future_final = m_final.make_future_dataframe(periods=3, freq="MS")
forecast_final = m_final.predict(future_final)
forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

print(forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))

In [0]:
fig = m_final.plot(forecast_final)
plt.title("Monthly Forecast — Full History + Next 3 Months")
plt.ylabel("Net Units")
plt.tight_layout()
plt.show()

In [0]:
naive_monthly_preds = train_monthly["y"].tail(3).mean()  # simple: predict last 3 months' average, repeated
naive_preds_array = [naive_monthly_preds] * 3

print(f"Monthly Naive Baseline WAPE: {wape(test_monthly['y'].values, naive_preds_array):.3%}")

In [0]:
brand_volume = silver.groupby("brand_code").agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count"),
    date_range_start=("posting_date", "min"),
    date_range_end=("posting_date", "max")
).reset_index()
print(brand_volume)

Brand-wise gold dataset

In [0]:
from src.transform.build_gold_features import build_gold_brand_weekly

silver_trimmed = silver[silver["posting_date"] <= pd.Timestamp("2026-06-12")].copy()

gold_brand_weekly = build_gold_brand_weekly(silver_trimmed)
print(gold_brand_weekly.shape)
gold_brand_weekly.tail(10)

Train the model

In [0]:
gold_brand_weekly["brand_code"] = gold_brand_weekly["brand_code"].astype("category")

feature_cols_brand = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "brand_code"]
target_col = "total_units_sold"

model_data_brand = gold_brand_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_brand = model_data_brand.sort_values("week_start")

split_idx = int(len(model_data_brand) * 0.8)
train_brand = model_data_brand.iloc[:split_idx]
test_brand = model_data_brand.iloc[split_idx:]

print(f"Train: {len(train_brand)} rows, Test: {len(test_brand)} rows")

X_train_b, y_train_b = train_brand[feature_cols_brand], train_brand[target_col]
X_test_b, y_test_b = test_brand[feature_cols_brand], test_brand[target_col]

Train and evaluate

In [0]:
with mlflow.start_run(run_name="phase2_brand_weekly"):
    params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6}
    mlflow.log_params(params)

    model_brand = lgb.LGBMRegressor(**params)
    model_brand.fit(X_train_b, y_train_b, categorical_feature=["brand_code"])

    preds_b = model_brand.predict(X_test_b)

    overall_wape = wape(y_test_b, preds_b)
    mlflow.log_metric("wape", overall_wape)
    mlflow.lightgbm.log_model(model_brand, name="model")

    print(f"Overall brand-model WAPE: {overall_wape:.3%}")

    test_brand_results = test_brand.copy()
    test_brand_results["prediction"] = preds_b
    for brand in test_brand_results["brand_code"].unique():
        subset = test_brand_results[test_brand_results["brand_code"] == brand]
        brand_wape = wape(subset["total_units_sold"], subset["prediction"])
        baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
        mlflow.log_metric(f"wape_{brand}", brand_wape)
        print(f"{brand} — Model WAPE: {brand_wape:.3%}   Baseline WAPE: {baseline_wape:.3%}   ({len(subset)} weeks)")

In [0]:
dagenite_weekly = gold_brand_weekly[gold_brand_weekly["brand_code"] == "DAGENITE"]
print(dagenite_weekly["total_units_sold"].describe())

In [0]:
final_brand_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_b = model_data_brand[feature_cols_brand]
y_all_b = model_data_brand[target_col]
final_brand_model.fit(X_all_b, y_all_b, categorical_feature=["brand_code"])

last_week_start = gold_brand_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

next_week_rows = []
for brand in ["EXIDE", "DAGENITE"]:
    brand_hist = gold_brand_weekly[gold_brand_weekly["brand_code"] == brand]
    lag_val = brand_hist[brand_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
    lag_val = lag_val[0] if len(lag_val) > 0 else None
    rolling_val = brand_hist["total_units_sold"].tail(4).mean()

    next_week_rows.append({
        "week_of_year": next_week_start.isocalendar()[1],
        "month": next_week_start.month,
        "contains_month_end": int(next_week_start.month != next_week_end.month),
        "lag_4w": lag_val,
        "rolling_avg_4w": rolling_val,
        "brand_code": brand,
    })

next_week_brand_df = pd.DataFrame(next_week_rows)
next_week_brand_df["brand_code"] = next_week_brand_df["brand_code"].astype("category")

predictions = final_brand_model.predict(next_week_brand_df[feature_cols_brand])
for brand, pred in zip(next_week_brand_df["brand_code"], predictions):
    print(f"{brand}: predicted {pred:.0f} net units for week starting {next_week_start.date()}")

Brand-wise monthly

In [0]:
from src.transform.build_gold_features import build_gold_brand_monthly

monthly_cutoff = pd.Timestamp("2026-05-31")
silver_trimmed_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()

gold_brand_monthly = build_gold_brand_monthly(silver_trimmed_monthly)
print(gold_brand_monthly.tail(10))

Model Train

In [0]:
brand_forecast_results = {}

for brand in ["EXIDE", "DAGENITE"]:
    brand_df = gold_brand_monthly[gold_brand_monthly["brand_code"] == brand][["month_start", "total_units_sold"]]
    brand_df = brand_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    train_b = brand_df.iloc[:-3]
    test_b = brand_df.iloc[-3:]

    m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m_test.fit(train_b)

    future_test = m_test.make_future_dataframe(periods=3, freq="MS")
    forecast_test = m_test.predict(future_test)
    test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

    model_wape = wape(test_b["y"].values, test_preds)
    naive_pred = train_b["y"].tail(3).mean()
    baseline_wape = wape(test_b["y"].values, [naive_pred] * 3)

    print(f"{brand} — Model WAPE: {model_wape:.3%}   Naive Baseline WAPE: {baseline_wape:.3%}")

    brand_forecast_results[brand] = {"train": train_b, "test": test_b, "full": brand_df}

3 months forecast

In [0]:
for brand, data in brand_forecast_results.items():
    m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m_final.fit(data["full"])

    future_final = m_final.make_future_dataframe(periods=3, freq="MS")
    forecast_final = m_final.predict(future_final)
    forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

    print(f"\n{brand} — 3-month forecast:")
    print(forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))

In [0]:
brand_df_dag = gold_brand_monthly[gold_brand_monthly["brand_code"] == "DAGENITE"][["month_start", "total_units_sold"]]
brand_df_dag = brand_df_dag.rename(columns={"month_start": "ds", "total_units_sold": "y"})

train_dag = brand_df_dag.iloc[:-3]
test_dag = brand_df_dag.iloc[-3:]

m_test_dag = Prophet(yearly_seasonality=False, weekly_seasonality=False)
m_test_dag.fit(train_dag)

future_test_dag = m_test_dag.make_future_dataframe(periods=3, freq="MS")
forecast_test_dag = m_test_dag.predict(future_test_dag)
test_preds_dag = forecast_test_dag.tail(3)["yhat"].clip(lower=0).values

print(f"DAGENITE (no yearly seasonality) WAPE: {wape(test_dag['y'].values, test_preds_dag):.3%}")

In [0]:
# DAGENITE monthly forecast: use naive baseline (3-month rolling average) —
# Prophet does not outperform this for DAGENITE (49.2%/41.5% WAPE vs 38.7% baseline)
dagenite_monthly_forecast = gold_brand_monthly[
    gold_brand_monthly["brand_code"] == "DAGENITE"
]["total_units_sold"].tail(3).mean()

print(f"DAGENITE naive 3-month forecast (flat, repeated): {dagenite_monthly_forecast:.0f} units/month")